In [1]:
import pandas as pd
import numpy as np
import pyarrow
from ml_utils.src import *
from ml_utils.config import *
import itertools
import gc 

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import  train_test_split
from sklearn.metrics import classification_report

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from keras.regularizers import l2


print(tf.config.list_physical_devices())
BATCH_SIZE = 64

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df_treino = pd.read_parquet(DATA_DIR, columns = colunas)
df_teste = pd.read_parquet("2023", columns = colunas)

## 1.2- Pré-Processando os Dados

In [3]:
df_treino = preparar_dados(df_treino, objetivo = '', n_samples = 400_000)
df_teste = preparar_dados(df_teste, objetivo = '', n_samples = 100_000)

## 1.3- Construção da Matriz X e Vetor y

In [4]:
X_treino = df_treino.drop([ 'FALTOU'], axis=1)
X_teste = df_teste.drop([ 'FALTOU'], axis=1)

y_treino = df_treino['FALTOU']
y_teste = df_teste['FALTOU']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_treino, y_treino, test_size=0.2, random_state=42)

## 1.5 - Tratando os Dados

In [6]:
preprocessador = pre_processor(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)


In [7]:
max_neurons = num_max_neuronio(X_train, d = X_train.shape[1])
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 192


## 1.6 - Estourando a rede

In [12]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1],)))

model.add(Dense(256, kernel_initializer='he_normal', 
                activation='relu'))

model.add(Dense(256, kernel_initializer='he_normal', 
                activation='relu'))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

# Compilar o modelo
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 256)            │        12,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,081 (305.00 KB)

 Trainable params: 78,081 (305.00 KB)

 Non-trainable params: 0 (0.00 B)

None


In [13]:
history = model.fit(X_train, y_train, epochs=100, batch_size = BATCH_SIZE)

Epoch 1/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7374 - loss: 0.5454
Epoch 2/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7436 - loss: 0.5343
Epoch 3/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7443 - loss: 0.5315
Epoch 4/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7447 - loss: 0.5295
Epoch 5/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7457 - loss: 0.5283
Epoch 6/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7459 - loss: 0.5272
Epoch 7/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7474 - loss: 0.5256
Epoch 8/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7480 - loss: 0.5240
Epoch 9/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7489 - loss: 0.5220
Epoch 10/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7501 - loss: 0.5200
Epoch 11/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7513 - loss: 0.5173
Epoch 12/100
1443/1

In [14]:
# Obtendo a acuracia no conjunto de treinamento
E_in, acc_train = model.evaluate(X_train, y_train, batch_size=BATCH_SIZE, verbose=0)

# Obtendo a acuracia no conjunto de validação
E_val, acc_val = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)

print(f"--> E_val - E_in = {E_val - E_in:.4f}")
print(f'--> Acuracia (treino): {acc_train:.4f}')
print(f'--> Acuracia (validação): {acc_val:.4f}')
print(f"--> acc_train - acc_val = {acc_train - acc_val:.4f}")

--> E_val - E_in = 1.7833
--> Acuracia (treino): 0.9240
--> Acuracia (validação): 0.6742
--> acc_train - acc_val = 0.2497


In [20]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1],)))

model.add(Dense(256, kernel_initializer='he_normal', kernel_regularizer=l2(0.001),
                activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(256, kernel_initializer='he_normal', kernel_regularizer=l2(0.001),
                activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

# Compilar o modelo
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │        12,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,081 (305.00 KB)

 Trainable params: 78,081 (305.00 KB)

 Non-trainable params: 0 (0.00 B)

None


In [21]:
history = model.fit(X_train, y_train, epochs=100, batch_size = BATCH_SIZE,
                    validation_data=(X_val, y_val),
                    verbose=1)

Epoch 1/100


1443/1443 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7261 - loss: 0.9444 - val_accuracy: 0.7463 - val_loss: 0.6702
Epoch 2/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7389 - loss: 0.6086 - val_accuracy: 0.7484 - val_loss: 0.5657
Epoch 3/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7399 - loss: 0.5608 - val_accuracy: 0.7483 - val_loss: 0.5410
Epoch 4/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7400 - loss: 0.5503 - val_accuracy: 0.7451 - val_loss: 0.5393
Epoch 5/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7391 - loss: 0.5483 - val_accuracy: 0.7459 - val_loss: 0.5352
Epoch 6/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7405 - loss: 0.5464 - val_accuracy: 0.7455 - val_loss: 0.5351
Epoch 7/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7398 - loss: 0.5464 - val_accuracy: 0.7462 - val_loss: 0.5354
Epoch 8/100
1443/1443 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7405 - loss: 0.5449 - val_

In [22]:
# Obtendo a acuracia no conjunto de treinamento
E_in, acc_train = model.evaluate(X_train, y_train, batch_size=BATCH_SIZE, verbose=0)

# Obtendo a acuracia no conjunto de validação
E_val, acc_val = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)

print(f"--> E_val - E_in = {E_val - E_in:.4f}")
print(f'--> Acuracia (treino): {acc_train:.4f}')
print(f'--> Acuracia (validação): {acc_val:.4f}')
print(f"--> acc_train - acc_val = {acc_train - acc_val:.4f}")

--> E_val - E_in = -0.0044
--> Acuracia (treino): 0.7436
--> Acuracia (validação): 0.7483
--> acc_train - acc_val = -0.0048


## 1.7 - Treinando a Rede Neural com melhores parâmetros

In [ ]:
param_grid = {
    'neurons':       [max_neurons - 1],
    'learning_rate': [0.01, 0.001, 0.0001],   
    'batch_size':    [64],             
    'epochs':        [100],
    'l2_reg':        [0.001, 0.01, 0.1],          
    'dropout':       [0.0, 0.2],
}

keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total de combinações: {len(combinacoes)}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = []

for params in combinacoes:
    print(f"Testando: {params}")
    
    accs = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

        classes = np.unique(y_tr)
        weights = compute_class_weight(
            class_weight='balanced',
            classes=classes,
            y=y_tr
        )

        class_weight = dict(zip(classes, weights))
        model = create_model(
            input_dim=X_train.shape[1],
            neurons=params['neurons'],
            learning_rate=params['learning_rate'],
            l2_reg=params['l2_reg'],
            dropout=params['dropout']
        )

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val_fold, y_val_fold),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            class_weight=class_weight,
            verbose=1
        )

        loss, acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        accs.append(acc)

        del model
        gc.collect()

    mean_acc = np.mean(accs)

    resultados.append({
        'params': params,
        'mean_accuracy': mean_acc
    })

    print(f"Accuracy média: {mean_acc:.4f}")

Total de combinações: 18
Testando: {'neurons': 191, 'learning_rate': 0.01, 'batch_size': 64, 'epochs': 100, 'l2_reg': 0.001, 'dropout': 0.0}
Epoch 1/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6210 - loss: 0.6725 - val_accuracy: 0.6361 - val_loss: 0.6300
Epoch 2/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6277 - loss: 0.6366 - val_accuracy: 0.5495 - val_loss: 0.6887
Epoch 3/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6305 - loss: 0.6336 - val_accuracy: 0.6576 - val_loss: 0.6141
Epoch 4/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6339 - loss: 0.6328 - val_accuracy: 0.6973 - val_loss: 0.5910
Epoch 5/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6323 - loss: 0.6336 - val_accuracy: 0.6804 - val_loss: 0.5957
Epoch 6/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6348 - loss: 0.6330 - val_accuracy: 0.6527 - val_loss: 0.6146
Epoch 7/100
1154/1154 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6308

In [24]:
resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.sort_values(by='mean_accuracy', ascending=False)

In [25]:
resultados_df.to_csv("resultados_presenca_socioeconomicos.csv", index=False)

In [8]:
resultados_df = pd.read_csv("resultados_presenca_socioeconomicos.csv")

In [9]:
resultados_df.params[0]

"{'neurons': 191, 'learning_rate': 0.01, 'batch_size': 64, 'epochs': 100, 'l2_reg': 0.1, 'dropout': 0.2}"

## Treinando o modelo com todos os dados e testando no ano de 2023

In [17]:
preprocessador = pre_processor(X_treino)

X_treino = preprocessador.transform(X_treino).astype(np.float32)
X_teste  = preprocessador.transform(X_teste).astype(np.float32)

In [18]:
adam = Adam(learning_rate=0.01)

model = Sequential()

model.add(Input(shape=(X_treino.shape[1],)))

model.add(Dense(191, kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(0.1), 
                activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Compilar o modelo
model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 191)            │         8,977 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 191)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,169 (35.82 KB)

 Trainable params: 9,169 (35.82 KB)

 Non-trainable params: 0 (0.00 B)

None


In [25]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_treino
)
class_weight = dict(zip(classes, weights))

history = model.fit(X_treino, y_treino,
                    epochs=100,
                    class_weight=class_weight,
                    batch_size=BATCH_SIZE) 

# Obtendo a acuracia no conjunto de treinamento
E_in, acc_train = model.evaluate(X_treino, y_treino, batch_size=BATCH_SIZE, verbose=0)

# Obtendo a acuracia no conjunto de teste
E_test, acc_test = model.evaluate(X_teste, y_teste, batch_size=BATCH_SIZE, verbose=0)

print(f"--> E_test - E_in = {E_test - E_in:.4f}")
print(f'--> Acuracia (treino): {acc_train:.4f}')
print(f'--> Acuracia (teste): {acc_test:.4f}')
print(f"--> acc_train - acc_test = {acc_train - acc_test:.4f}")

Epoch 1/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6330 - loss: 0.6733
Epoch 2/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 984us/step - accuracy: 0.6354 - loss: 0.6725
Epoch 3/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 996us/step - accuracy: 0.6343 - loss: 0.6713
Epoch 4/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 977us/step - accuracy: 0.6371 - loss: 0.6710
Epoch 5/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6343 - loss: 0.6732
Epoch 6/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 951us/step - accuracy: 0.6364 - loss: 0.6709
Epoch 7/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 953us/step - accuracy: 0.6337 - loss: 0.6714
Epoch 8/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 959us/step - accuracy: 0.6348 - loss: 0.6706
Epoch 9/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6396 - loss: 0.6723  
Epoch 10/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6368 - loss: 0.6722  
Epoch 11/100
1803/1803 ━━━━━━━━━━━━━━━━━━━━ 2s 984us/step - accuracy: 0.6327 - loss: 0.6719
E

In [26]:
y_pred_proba = model.predict(X_teste)
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_teste, y_pred))

1062/1062 ━━━━━━━━━━━━━━━━━━━━ 1s 488us/step
              precision    recall  f1-score   support

           0       0.80      0.82      0.81     25672
           1       0.41      0.38      0.39      8310

    accuracy                           0.71     33982
   macro avg       0.61      0.60      0.60     33982
weighted avg       0.71      0.71      0.71     33982

